In [1]:
from model import Retriever
import pandas as pd
import os
import json
import warnings
warnings.filterwarnings('ignore')
retriever = Retriever('cambridgeltl/SapBERT-from-PubMedBERT-fulltext')
retriever.load_dictionary_all('./umls_dictionary.txt')
retriever.load_dictionary_bodyloc('./umls_body_loc_dictionary.txt')
retriever.embed_dictionary(32768)
retriever.faiss_setup()

Checking if cache file exists at ./cache//dense_embed_all.pt
Cache file found. Loading dense embeddings from cache.
Checking if cache file exists at ./cache//dense_embed_bodyloc.pt
Cache file found. Loading dense embeddings from cache.


In [37]:
def eval_recall_precision(annotations_gold, eval_dfs):

    ner_labels = annotations_gold['code'].tolist()
    ner_labels = [list(json.loads(item).keys())[0] for item in ner_labels]
    annotations_gold['code'] = ner_labels
    ner_predic = eval_dfs['code'].tolist()
    ner_predic = [list(json.loads(item).keys())[0] for item in ner_predic if type(item) == str]
    eval_dfs['code'] = ner_predic
    # print(annotations_gold['phrase'].tolist())
    # print(eval_dfs['mention'].tolist())
    # print(annotations_gold['code'].tolist())
    # print(eval_dfs['code'].tolist())
    
    ner_correct = 0
    correct_phrases = []
    for item in ner_predic:
        if item in ner_labels:
            ner_correct += 1
            ner_labels.remove(item)
            correct_phrases.append(item)
    recall = ner_correct / len(annotations_gold['phrase'])
    precision = ner_correct / len(eval_dfs['mention'])

    # recall = len(set(ner_labels).intersection(ner_predic)) / len(set(ner_labels))
    # precision = len(set(ner_labels).intersection(ner_predic)) / len(set(ner_predic))
    print('NER recall:\t', recall)
    print('NER precision:\t', precision)

    
    columns = ['assertion_status']
    df_predic = eval_dfs[eval_dfs['code'].isin(correct_phrases)]
    df_labels = annotations_gold[annotations_gold['code'].isin(correct_phrases)]
    for col in columns:
        labels, predic = [], []
        for I, P in zip(df_labels[col].tolist(),df_labels['code'].tolist()):
    
            if col == 'assertion_status':
                # print(I.lower(), I.lower() in ['possible', 'conditional', 'hypothetical'])
                if I.lower() in ['possible', 'conditional', 'hypothetical']:
                    I = 'Speculative'
                elif I.lower() in ['title', 'Not associated with the patient']:
                    I = 'Not Associated'
            if I not in ["not applicable", 'null', 'nan'] and not pd.isna(I):
                try:
                    I = str(float(I))
                except:
                    I = str(I)
                labels.append((I+'_'+P).lower())
        for I, P in zip(df_predic[col].tolist(),df_predic['code'].tolist()):
            if I not in ["not applicable", 'null', 'nan'] and not pd.isna(I):
                try:
                    I = str(float(I))
                except:
                    I = str(I)
                predic.append((I+'_'+P).lower())

        
        label_number = len(labels)
        predi_number = len(predic)
        print('labs', labels)
        print('pred', predic)
        correct = 0
        for item in predic:
            if item in labels:
                correct += 1
                labels.remove(item)
        recall = correct / (label_number+1e-10)
        precision = correct / (predi_number+1e-10)
        print(f'{col} recall:\t', recall, label_number)
        print(f'{col} precision:\t', precision, predi_number)
    if label_number <= 2:
        return None
    return 2 * recall * precision / (precision + recall + 1e-10)

In [38]:
### llama 3 
annotation = []
path = '../ehrllm/0821_model/structure_eval/annotations/MGB/'
annotation_file = [item for item in os.listdir(path) if not item.startswith('.')]
dfs = []
for fi in annotation_file:
    dfs.append(pd.read_csv(path + fi))
    dfs[-1].rename(columns={'Label': 'label', 'Labels': 'label', 'labels': 'label'}, inplace=True)
dfs = pd.concat(dfs)
annotations_gold = dfs[dfs['label']!=3]
all_code = retriever.embedding_retrieval_all(annotations_gold['phrase'].tolist())
loc_code = retriever.embedding_retrieval_bodyloc([item if not pd.isna(item) else None for item in annotations_gold['body_location'].tolist()])
# all_code = [list(item.keys())[0] if item is not None else None for item in all_code]
# loc_code = [list(item.keys())[0] if item is not None else None for item in loc_code]
annotations_gold['code'] = all_code
annotations_gold['body_location_code'] = loc_code


path = '/n/data1/hsph/biostat/celehs/lab/hongyi/language-into-clinical-data/mimic_pred/llama3/outputs/'
eval_file = ['0.csv', '19.csv', '38.csv', '57.csv']
eval_dfs = []
for fi in eval_file:
    print(fi, path)
    eval_dfs.append(pd.read_csv(path + fi))
eval_dfs = pd.concat(eval_dfs)

eval_recall_precision(annotations_gold, eval_dfs)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 32.52it/s]


0.csv /n/data1/hsph/biostat/celehs/lab/hongyi/language-into-clinical-data/mimic_pred/llama3/outputs/
19.csv /n/data1/hsph/biostat/celehs/lab/hongyi/language-into-clinical-data/mimic_pred/llama3/outputs/
38.csv /n/data1/hsph/biostat/celehs/lab/hongyi/language-into-clinical-data/mimic_pred/llama3/outputs/
57.csv /n/data1/hsph/biostat/celehs/lab/hongyi/language-into-clinical-data/mimic_pred/llama3/outputs/
NER recall:	 0.4936708860759494
NER precision:	 0.48148148148148145
labs ['present_c0728940', 'present_c0023269', 'present_c0184918', 'present_c0023269', 'present_c0017168', 'present_c0040264', 'present_c0009376', 'present_c1378703', 'present_c1144661', 'present_c0020538', 'present_c0855057', 'present_c0024198', 'present_c0020264', 'speculative_c0003864', 'present_c0065374', 'present_c0301532', 'present_c0008318', 'present_c0008318', 'speculative_c4316895', 'absent_c0019214', 'absent_c0578736', 'present_c0202823', 'present_c0027651', 'present_c0812425', 'present_c0728940', 'present_c455

0.7441860464600053

In [39]:

annotation = []
path = '../ehrllm/0821_model/structure_eval/annotations/MGB/'
annotation_file = [item for item in os.listdir(path) if not item.startswith('.')]
dfs = []
for fi in annotation_file:
    dfs.append(pd.read_csv(path + fi))
    dfs[-1].rename(columns={'Label': 'label', 'Labels': 'label', 'labels': 'label'}, inplace=True)
dfs = pd.concat(dfs)
annotations_gold = dfs[dfs['label']!=3]
all_code = retriever.embedding_retrieval_all(annotations_gold['phrase'].tolist())
loc_code = retriever.embedding_retrieval_bodyloc([item if not pd.isna(item) else None for item in annotations_gold['body_location'].tolist()])
# all_code = [list(item.keys())[0] if item is not None else None for item in all_code]
# loc_code = [list(item.keys())[0] if item is not None else None for item in loc_code]
annotations_gold['code'] = all_code
annotations_gold['body_location_code'] = loc_code


path = '../ehrllm/0821_model/annotation_ood/'
eval_file = [item for item in os.listdir(path) if not item.startswith('.') and item.isdigit()]
eval_dfs = []
for fi in eval_file:
    print(fi, path)
    eval_dfs.append(pd.read_csv(path + fi + '/gpt4o_model_result.csv'))
eval_dfs = pd.concat(eval_dfs)

# print(eval_dfs[['code', 'mention']])
# print(annotations_gold[['code', 'phrase']])
# eval_dfs.to_csv('pred.csv')
# annotations_gold.to_csv('refs.csv')
eval_recall_precision(annotations_gold, eval_dfs)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 36.56it/s]


0 ../ehrllm/0821_model/annotation_ood/
19 ../ehrllm/0821_model/annotation_ood/
38 ../ehrllm/0821_model/annotation_ood/
57 ../ehrllm/0821_model/annotation_ood/
NER recall:	 0.5569620253164557
NER precision:	 0.5789473684210527
labs ['present_c0030193', 'present_c0728940', 'present_c0023269', 'present_c1709603', 'present_c0184918', 'present_c0023269', 'present_c1709157', 'present_c0017168', 'present_c0040264', 'present_c1384641', 'present_c0009376', 'associated_with_someone_else_c0007102', 'present_c1378703', 'present_c1144661', 'present_c0020538', 'present_c0855057', 'present_c0375983', 'present_c0024198', 'present_c0020264', 'present_c0000970', 'speculative_c0003864', 'present_c0065374', 'present_c0301532', 'present_c0008318', 'present_c0008318', 'speculative_c0030842', 'speculative_c4316895', 'absent_c0019214', 'absent_c0577559', 'present_c0202823', 'absent_c0006826', 'present_c0812425', 'present_c0728940', 'present_c1337016', 'present_c4551687', 'present_c2733303', 'present_c0475440'

0.826086956469967

In [40]:

pred_path = '/n/data1/hsph/biostat/celehs/lab/hongyi/language-into-clinical-data/mimic_pred/llama3/outputs/'
files = os.listdir(pred_path)
files = [item for item in files if item.endswith('csv') if not item[0].isdigit()]
# print(files)
path = '../ehrllm/0821_model/structure_eval/annotations/'
overall_f1 = {}

for fi in files:
    prefix = fi.rstrip('_model_result.csv')
    if 'Hongyi' in prefix:
        dfs = pd.read_csv(path + f'{prefix}.csv')
    else:
        try:
            dfs = pd.read_csv(path + f'Enci_{prefix}.csv')
        except:
            dfs = pd.read_csv(path + f'Mo_{prefix}.csv')
    dfs.rename(columns={'Label': 'label', 'Labels': 'label', 'labels': 'label', 'Unnamed: 9': 'label'}, inplace=True)
    annotations_gold = dfs[dfs['label']!=3]
    all_code = retriever.embedding_retrieval_all(annotations_gold['phrase'].tolist())
    loc_code = retriever.embedding_retrieval_bodyloc([item if not pd.isna(item) else None for item in annotations_gold['body_location'].tolist()])
    annotations_gold['code'] = all_code
    if len(loc_code) == 0:
        loc_code = [None] * len(annotations_gold['code'])
    annotations_gold['body_location_code'] = loc_code
    
    eval_dfs = pd.read_csv(pred_path + fi)
    res = eval_recall_precision(annotations_gold, eval_dfs)
    if res is not None:
        overall_f1[fi] = res
import numpy as np
print(np.mean(list(overall_f1.values())))
print(overall_f1)


  0%|                                                                                                                                                                                                                  | 0/1 [00:00<?, ?it/s]

NER recall:	 0.625
NER precision:	 0.7142857142857143
labs ['absent_c0240859', 'absent_c0043144', 'absent_c0035508', 'absent_c0019214', 'present_c0013491', 'absent_c0149651', 'absent_c0010520', 'absent_c0013604', 'present_c0011620', 'absent_c0041582', 'absent_c2004491', 'present_c0918147', 'speculative_c0034065', 'present_c0039985', 'present_c3165806']
pred ['absent_c0240859', 'absent_c0043144', 'absent_c0035508', 'absent_c0019214', 'present_c0013491', 'absent_c0149651', 'absent_c0010520', 'absent_c0013604', 'present_c0011620', 'absent_c0041582', 'present_c2004491', 'present_c0918147', 'speculative_c0034065', 'present_c0039985', 'present_c3165806']
assertion_status recall:	 0.9333333333271111 15
assertion_status precision:	 0.9333333333271111 15
NER recall:	 0.41379310344827586
NER precision:	 0.6
labs ['present_c0740341', 'present_c0043031', 'present_c0525032', 'present_c3873974', 'present_c0525032', 'present_c0042879', 'present_c0016709', 'present_c0525032', 'absent_c0048470', 'prese

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 57.49it/s]


NER recall:	 0.7
NER precision:	 0.6666666666666666
labs ['present_c0176751', 'present_c0176813', 'present_c0179740', 'present_c0041281', 'present_c0034977', 'present_c0025853', 'present_c0034977', 'present_c0041281', 'present_c1145670', 'present_c0518239', 'present_c0181760', 'present_c0030054', 'present_c0036537', 'present_c0036537', 'present_c0343401']
pred ['present_c0176751', 'present_c0176813', 'present_c0179740', 'present_c0176813', 'present_c0041281', 'absent_c0025853', 'present_c0034977', 'present_c0041281', 'present_c1145670', 'present_c0518239', 'present_c0181760', 'present_c0030054', 'present_c0036537', 'present_c0036537', 'present_c0343401']
assertion_status recall:	 0.8666666666608889 15
assertion_status precision:	 0.8666666666608889 15
NER recall:	 0.6
NER precision:	 0.5
labs ['present_c0036078', 'present_c0086787', 'present_c0037265']
pred ['present_c0036078', 'present_c0086787', 'present_c0037265']
assertion_status recall:	 0.9999999999666667 3
assertion_status preci

  0%|                                                                                                                                                                                                                  | 0/1 [00:00<?, ?it/s]

...

NER recall:	 0.47619047619047616
NER precision:	 0.45454545454545453
labs ['present_c0549111', 'absent_c1285498', 'present_c0507851', 'absent_c2936350', 'present_c0026018', 'present_c0021044', 'present_c0030664', 'present_c2827061', 'present_c4505475', 'present_c0037265']
pred ['present_c0549111', 'absent_c1285498', 'present_c0507851', 'absent_c2936350', 'present_c0026018', 'present_c0021044', 'present_c0030664', 'present_c2827061', 'present_c4505475', 'present_c0037265']
assertion_status recall:	 0.99999999999 10
assertion_status precision:	 0.99999999999 10
NER recall:	 0.5
NER precision:	 1.0
labs ['present_c0039985', 'present_c0039985']
pred ['speculative_c0039985']
assertion_status recall:	 0.0 2
assertion_status precision:	 0.0 1
NER recall:	 0.6842105263157895
NER precision:	 0.6190476190476191
labs ['present_c0040165', 'present_c0282139', 'present_c0060926', 'speculative_c0699142', 'speculative_c0591416', 'present_c0286651', 'present_c0082607', 'present_c0296800', 'present_c072


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 50.11it/s]


NER recall:	 0.8
NER precision:	 0.6666666666666666
labs ['present_c0935966', 'present_c0701094', 'present_c0026549', 'present_c0086787', 'present_c0795585', 'absent_c1272738', 'present_c0525032', 'present_c0699129']
pred ['present_c0935966', 'present_c0701094', 'present_c0026549', 'present_c0086787', 'present_c0795585', 'absent_c1272738', 'present_c0525032', 'present_c0699129']
assertion_status recall:	 0.9999999999875 8
assertion_status precision:	 0.9999999999875 8
NER recall:	 0.7222222222222222
NER precision:	 0.7222222222222222
labs ['speculative_c0000731', 'speculative_c0042963', 'present_c3879329', 'present_c0699129', 'present_c0699129', 'present_c0699129', 'present_c0525032', 'speculative_c0723148', 'present_c0700776', 'present_c0876064', 'present_c0282386', 'present_c0002598', 'present_c0286036']
pred ['speculative_c0000731', 'speculative_c0042963', 'present_c3879329', 'present_c0699129', 'absent_c0699129', 'present_c0699129', 'present_c0525032', 'present_c0723148', 'present_

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 47.16it/s]

NER recall:	 0.8
NER precision:	 0.5714285714285714
labs ['speculative_c0086787', 'absent_c0199782', 'absent_c0557977', 'present_c0392201']
pred ['present_c0086787', 'absent_c0199782', 'absent_c0557977', 'present_c0392201']
assertion_status recall:	 0.74999999998125 4
assertion_status precision:	 0.74999999998125 4
NER recall:	 0.6666666666666666
NER precision:	 0.5
labs ['present_c0042963', 'speculative_c0281936', 'present_c0412620', 'present_c0677554', 'present_c0267771', 'present_c0400847', 'present_c0801760', 'present_c0743973', 'present_c0700292', 'present_c0003232', 'present_c0009378', 'present_c2348535', 'present_c0235329', 'present_c0085704', 'present_c0032285', 'present_c0002915', 'present_c0021925', 'absent_c0009378', 'absent_c2348535', 'present_c0179740']
pred ['present_c0042963', 'speculative_c0281936', 'present_c0412620', 'present_c0677554', 'present_c0267771', 'present_c0400847', 'present_c0801760', 'present_c0743973', 'present_c0700292', 'present_c0003232', 'speculative_

In [41]:

files = os.listdir('./mimic_pred/')
files = [item for item in files if item.endswith('csv')]
# print(files)
path = '../ehrllm/0821_model/structure_eval/annotations/'
overall_f1 = {}

for fi in files:
    prefix = fi.rstrip('_model_result.csv')
    # if fi != 'Enci_EHRSTRUCT_59589-59110-0_0_model_result.csv':
    #     continue
    dfs = pd.read_csv(path + f'{prefix}.csv')
    dfs.rename(columns={'Label': 'label', 'Labels': 'label', 'labels': 'label', 'Unnamed: 9': 'label'}, inplace=True)
    annotations_gold = dfs[dfs['label']!=3]
    all_code = retriever.embedding_retrieval_all(annotations_gold['phrase'].tolist())
    loc_code = retriever.embedding_retrieval_bodyloc([item if not pd.isna(item) else None for item in annotations_gold['body_location'].tolist()])
    annotations_gold['code'] = all_code
    if len(loc_code) == 0:
        loc_code = [None] * len(annotations_gold['code'])
    annotations_gold['body_location_code'] = loc_code
    
    eval_dfs = pd.read_csv('./mimic_pred/' + fi)

    res = eval_recall_precision(annotations_gold, eval_dfs)
    if res is not None:
        overall_f1[fi] = res
import numpy as np
print(np.mean(list(overall_f1.values())))
print(overall_f1)


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 48.13it/s]

NER recall:	 0.5416666666666666
NER precision:	 0.8125
labs ['absent_c0240859', 'absent_c0043144', 'absent_c0035508', 'absent_c0019214', 'present_c0013491', 'present_c0011620', 'present_c0918147', 'speculative_c0034065', 'present_c0039985', 'present_c0412620', 'present_c3179130', 'present_c0879373', 'present_c3165806']
pred ['absent_c0240859', 'absent_c0043144', 'absent_c0035508', 'absent_c0019214', 'present_c0013491', 'present_c0011620', 'present_c0918147', 'present_c0918147', 'speculative_c0034065', 'present_c0039985', 'present_c0412620', 'present_c3179130', 'present_c0879373', 'present_c3165806']
assertion_status recall:	 0.9999999999923077 13
assertion_status precision:	 0.9285714285647959 14
NER recall:	 0.7241379310344828
NER precision:	 0.7777777777777778
labs ['not associated_c0009450', 'present_c0740341', 'present_c0489941', 'present_c0489941', 'present_c0699129', 'present_c0043031', 'present_c0525032', 'present_c3873974', 'present_c0525032', 'present_c0042879', 'present_c0016


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 49.75it/s]

NER recall:	 0.35
NER precision:	 0.5
labs ['present_c0179740', 'present_c0034977', 'present_c0034977', 'present_c1145670', 'present_c0181760', 'present_c0343401', 'present_c0000990']
pred ['present_c0179740', 'absent_c0034977', 'present_c0034977', 'present_c1145670', 'present_c0181760', 'present_c0343401', 'present_c0000990']
assertion_status recall:	 0.8571428571306122 7
assertion_status precision:	 0.8571428571306122 7
NER recall:	 0.9473684210526315
NER precision:	 0.8571428571428571
labs ['present_c0040165', 'present_c0282139', 'present_c0060926', 'speculative_c0699142', 'speculative_c0591416', 'present_c0286651', 'present_c0021461', 'present_c1254240', 'present_c0082607', 'present_c0296800', 'present_c0723783', 'present_c0065374', 'present_c0081876', 'speculative_c3485411', 'speculative_c0489941', 'present_c0042314', 'speculative_c0489941', 'present_c0011946']
pred ['present_c0040165', 'present_c0282139', 'present_c0060926', 'present_c0699142', 'present_c0591416', 'present_c02866


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 53.03it/s]


...

NER recall:	 0.8
NER precision:	 0.5714285714285714
labs ['speculative_c0086787', 'absent_c0199782', 'absent_c0557977', 'present_c0392201']
pred ['not associated_c0086787', 'absent_c0199782', 'absent_c0557977', 'present_c0392201']
assertion_status recall:	 0.74999999998125 4
assertion_status precision:	 0.74999999998125 4
NER recall:	 0.6666666666666666
NER precision:	 0.5
labs ['present_c0042963', 'present_c4084776', 'present_c4284399', 'speculative_c0281936', 'speculative_c1258215', 'present_c0412620', 'present_c0267771', 'present_c0400847', 'present_c0801760', 'present_c0743973', 'present_c0700292', 'present_c0003232', 'present_c0235329', 'present_c0028778', 'present_c0085704', 'present_c0032285', 'present_c0002915', 'present_c0021925', 'present_c0179740', 'present_c0030548']
pred ['present_c0042963', 'present_c4084776', 'present_c4284399', 'speculative_c0281936', 'speculative_c1258215', 'present_c0412620', 'present_c0267771', 'present_c0400847', 'present_c0801760', 'present_c074397

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 48.15it/s]

NER recall:	 0.8
NER precision:	 0.6666666666666666
labs ['present_c0036078', 'present_c0086787', 'present_c0444192', 'present_c0037265']
pred ['not associated_c0036078', 'not associated_c0086787', 'present_c0444192', 'speculative_c0037265', 'speculative_c0037265']
assertion_status recall:	 0.24999999999375 4
assertion_status precision:	 0.199999999996 5
NER recall:	 0.36666666666666664
NER precision:	 0.5
labs ['present_c1417325', 'present_c0102118', 'present_c0038317', 'present_c2748208', 'speculative_c0037265', 'present_c0020043', 'present_c0412594', 'present_c3916853', 'present_c0266968', 'speculative_c0029443', 'speculative_c3810196']
pred ['present_c1417325', 'present_c0102118', 'present_c0038317', 'speculative_c2748208', 'present_c2748208', 'speculative_c0037265', 'speculative_c0037265', 'present_c0020043', 'present_c0412594', 'present_c3916853', 'not associated_c0266968', 'speculative_c0029443', 'speculative_c3810196']
assertion_status recall:	 0.9090909090826447 11
assertion_s


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 49.86it/s]


NER recall:	 0.7647058823529411
NER precision:	 0.7222222222222222
labs ['present_c0038774', 'present_c0162712', 'present_c0000737', 'present_c0020545', 'present_c0020538', 'present_c4255010', 'present_c3537184', 'present_c1322812', 'present_c0018099', 'present_c0010346', 'present_c0562508', 'absent_c4084784', 'absent_c0009806']
pred ['not associated_c0038774', 'not associated_c0162712', 'present_c0000737', 'present_c0020545', 'present_c0020538', 'present_c4255010', 'present_c3537184', 'present_c1322812', 'present_c0018099', 'present_c0010346', 'present_c0562508', 'absent_c4084784', 'absent_c0009806']
assertion_status recall:	 0.8461538461473372 13
assertion_status precision:	 0.8461538461473372 13
NER recall:	 0.5238095238095238
NER precision:	 0.7333333333333333
labs ['present_c0549111', 'present_c1533591', 'absent_c1285498', 'present_c0374082', 'present_c0507851', 'absent_c2936350', 'absent_c0012737', 'present_c0021044', 'present_c0030664', 'present_c2827061', 'present_c0037265']
pr

In [31]:
overall_f1

{'Enci_EHRSTRUCT_58583-58958-1_1_model_result.csv': 0.649999999952,
 'Enci_EHRSTRUCT_58587-58922-1_1_model_result.csv': 0.7499999999500637,
 'Enci_EHRSTRUCT_58621-59013-0_0_model_result.csv': 0.41176470583391,
 'Enci_EHRSTRUCT_59163-59480-1_0_model_result.csv': 0.899999999950125,
 'Enci_EHRSTRUCT_59176-59367-0_0_model_result.csv': 0.7999999999500002,
 'Enci_EHRSTRUCT_59589-59110-0_0_model_result.csv': 0.857142857092898,
 'Enci_EHRSTRUCT_59594-59599-0_0_model_result.csv': 0.6666666666180555,
 'Enci_EHRSTRUCT_59606-59637-3_0_model_result.csv': 0.5714285713795919,
 'Mo_EHRSTRUCT_58675-58993-0_0_model_result.csv': 0.7272727272231404,
 'Mo_EHRSTRUCT_58689-59018-0_1_model_result.csv': 0.42307692302810646,
 'Mo_EHRSTRUCT_58711-59004-0_0_model_result.csv': 0.7428571428071835,
 'Mo_EHRSTRUCT_59003-58721-1_0_model_result.csv': 0.6111111110624999,
 'Mo_EHRSTRUCT_59094-59331-1_2_model_result.csv': 0.799999999952}